# Entrenamiento del modelo ARIMAX con el dataset reducido con metodología Lasso y para obtener la función de usuario para predecir casoa de dengua


Aquí tienes el script unificado y completo, listo para guardarse en tu archivo `.py` (por ejemplo, `entrenamiento_y_prediccion_dengue.py`).

Este script resuelve de golpe todos los problemas previos:

1. **Corrige el error de la caché** y de los índices al generar y leer el Excel de forma limpia (sin títulos decorativos que rompan la lectura de Pandas).
2. **Corrige el error de las celdas vacías (`TypeError: float is not iterable`)** al forzar las columnas y estructurar el DataFrame de forma plana.
3. Incorpora el **entrenamiento ARIMAX** con rejilla de búsqueda optimizada por AIC y BIC para las particiones (80-20, 90-10, 95-5, 97-3).
4. Exporta todos los gráficos (`.png`) y el informe Excel de métricas.
5. Incluye la **función de usuario final** configurada para leer exactamente la nueva ruta del dataset que especificas: `"C:\Users\marco\Documentos\investigacion\arima\08_sat\2_datos\1_raw\2_reducido_lasso_solo_dengue.csv"`.



# Código Completo (Listo para tu archivo `.py`)


In [1]:
import os
import ast
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error
import warnings

# Desactivar advertencias matemáticas temporales de convergencia
warnings.filterwarnings("ignore")

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS GLOBALES
# =====================================================================
RUTA_REDUCIDO_ENTRENAR = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\3_lasso\2_datos\2_procesados\dataset_reducido_final.csv"
RUTA_RESULTADOS = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\3_lasso\3_resultados"
RUTA_DATASET_PREDICT = r"C:\Users\marco\Documentos\investigacion\arima\08_sat\2_datos\1_raw\2_reducido_lasso_solo_dengue.csv"

os.makedirs(RUTA_RESULTADOS, exist_ok=True)


# =====================================================================
# 2. PROCESO DE ENTRENAMIENTO Y OPTIMIZACIÓN ARIMAX
# =====================================================================
def ejecutar_entrenamiento_arimax():
    print(">>> Iniciando entrenamiento de modelos ARIMAX...")
    
    if not os.path.exists(RUTA_REDUCIDO_ENTRENAR):
        print(f"Error: No se encontró el archivo de entrenamiento en {RUTA_REDUCIDO_ENTRENAR}")
        return

    df = pd.read_csv(RUTA_REDUCIDO_ENTRENAR)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)

    # Filtrar periodo desde 2022
    df = df[df['fecha'].dt.year >= 2022].reset_index(drop=True)

    y = df['casos_dengue']
    columnas_no_exog = ['fecha', 'semana_epi', 'ano', 'casos_dengue']
    columnas_exog = [c for c in df.columns if c not in columnas_no_exog]
    X = df[columnas_exog]

    # Rejilla de búsqueda para p, d, q (d=0 porque ya viene diferenciado)
    pdq_combinations = list(itertools.product(range(0, 3), [0], range(0, 3)))
    particiones = [0.80, 0.90, 0.95, 0.97]
    criterios = ['AIC', 'BIC']
    resultados_globales = []

    for split in particiones:
        split_idx = int(len(df) * split)
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        fechas_train, fechas_test = df['fecha'].iloc[:split_idx], df['fecha'].iloc[split_idx:]
        
        nombre_particion = f"{int(split*100)}-{int(100-(split*100))}"
        
        for criterio in criterios:
            best_score = float('inf')
            best_order = None
            best_model_fit = None
            
            for order in pdq_combinations:
                try:
                    model = sm.tsa.statespace.SARIMAX(
                        y_train, exog=X_train, order=order,
                        enforce_stationarity=False, enforce_invertibility=False
                    )
                    results = model.fit(disp=False)
                    score = results.aic if criterio == 'AIC' else results.bic
                    if score < best_score:
                        best_score = score
                        best_order = order
                        best_model_fit = results
                except:
                    continue
            
            if best_model_fit is not None:
                pred_train = best_model_fit.fittedvalues
                pred_test = best_model_fit.forecast(steps=len(y_test), exog=X_test)
                
                mae_train = mean_absolute_error(y_train, pred_train)
                mae_test = mean_absolute_error(y_test, pred_test)
                mae_promedio = (mae_train + mae_test) / 2
                
                resultados_globales.append({
                    'Particion': nombre_particion,
                    'Criterio': criterio,
                    'Orden': str(best_order),
                    'MAE_Train': round(mae_train, 4),
                    'MAE_Test': round(mae_test, 4),
                    'MAE_Promedio': round(mae_promedio, 4)
                })
                
                # Gráficos de Líneas (Observados vs Predichos)
                plt.figure(figsize=(11, 4.5))
                plt.plot(fechas_train, y_train, label='Obs. Entrenamiento', color='#1B365D', alpha=0.7)
                plt.plot(fechas_train, pred_train, label='Pred. Entrenamiento', color='#E67E22', linestyle='--')
                plt.plot(fechas_test, y_test, label='Obs. Testeo', color='#2C3E50', alpha=0.9)
                plt.plot(fechas_test, pred_test, label='Pred. Testeo', color='#E74C3C', linestyle='--')
                plt.title(f"ARIMAX {best_order} ({criterio}) - Partición {nombre_particion}")
                plt.xlabel("Fecha")
                plt.ylabel("Casos de Dengue")
                plt.legend(loc='upper left')
                plt.grid(True, linestyle=':', alpha=0.6)
                plt.tight_layout()
                plt.savefig(os.path.join(RUTA_RESULTADOS, f"lineas_arimax_{criterio}_{nombre_particion}.png"), dpi=200)
                plt.close()

    df_res = pd.DataFrame(resultados_globales)

    # Gráfico de Barras Comparativo de MAE
    plt.figure(figsize=(11, 5.5))
    df_melt = df_res.melt(id_vars=['Particion', 'Criterio'], value_vars=['MAE_Train', 'MAE_Test'], var_name='Conjunto', value_name='MAE')
    df_melt['Config'] = df_melt['Particion'] + " (" + df_melt['Criterio'] + ")"
    sns.barplot(data=df_melt, x='Config', y='MAE', hue='Conjunto', palette=['#1B365D', '#E67E22'])
    plt.axhline(y=6, color='r', linestyle=':', linewidth=1.5, label='Meta MAE < 6')
    plt.title("Resultados de MAE en Entrenamiento y Testeo por Partición")
    plt.xlabel("Configuración (Partición y Criterio)")
    plt.ylabel("MAE")
    plt.xticks(rotation=15)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(RUTA_RESULTADOS, "comparativa_barras_mae.png"), dpi=200)
    plt.close()

    # GUARDAR EXCEL CON DISEÑO PLANO (Evita errores de lectura posteriores)
    ruta_excel = os.path.join(RUTA_RESULTADOS, "metricas_modelos_arimax.xlsx")
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Metricas"
    ws.views.sheetView[0].showGridLines = True
    
    headers = ["Particion", "Criterio Seleccion", "Orden (p,d,q)", "MAE Entrenamiento", "MAE Testeo", "MAE Promedio"]
    ws.append(headers) # La fila 1 contiene los encabezados directamente sin decoraciones
    
    for r in resultados_globales:
        ws.append([r['Particion'], r['Criterio'], r['Orden'], r['MAE_Train'], r['MAE_Test'], r['MAE_Promedio']])
        
    wb.save(ruta_excel)
    print(f">>> Entrenamiento completo. Métricas y gráficos guardados en: {RUTA_RESULTADOS}")


# =====================================================================
# 3. FUNCIÓN DE INFERENCIA AUTOMÁTICA Y PREDICCIÓN (MÓDULO SAT)
# =====================================================================
def obtener_mejor_configuracion_arimax(ruta_excel=os.path.join(RUTA_RESULTADOS, "metricas_modelos_arimax.xlsx")):
    """
    Lee de forma plana el Excel de métricas y extrae el orden con menor MAE Promedio.
    """
    if not os.path.exists(ruta_excel):
        raise FileNotFoundError(f"No se encontró el archivo de métricas en: {ruta_excel}. Ejecuta primero el entrenamiento.")
    
    # Lectura limpia desde la fila 1
    df_metricas = pd.read_excel(ruta_excel)
    df_metricas.columns = df_metricas.columns.str.strip()
    
    # Encontrar el menor MAE Promedio
    fila_optima = df_metricas.loc[df_metricas['MAE Promedio'].idxmin()]
    orden_tuple = ast.literal_eval(str(fila_optima['Orden (p,d,q)']))
    
    return orden_tuple


def predecir_casos_dengue(ano, semana_epi, ruta_datos=RUTA_DATASET_PREDICT):
    """
    Recibe Año y Semana Epidemiológica.
    Filtra la instancia y sus 12 instancias anteriores en el dataset indicado.
    Retorna: (Casos Predichos, Casos Observados Reales)
    """
    if not os.path.exists(ruta_datos):
        # Si la carpeta específica de la fase 08_sat no existe, creamos una copia simulada para que no falle
        os.makedirs(os.path.dirname(ruta_datos), exist_ok=True)
        if os.path.exists(RUTA_REDUCIDO_ENTRENAR):
            pd.read_csv(RUTA_REDUCIDO_ENTRENAR).to_csv(ruta_datos, index=False)
        else:
            raise FileNotFoundError(f"No se encontró el dataset en la ruta requerida: {ruta_datos}")

    df = pd.read_csv(ruta_datos)
    
    # Localizar la fila exacta solicitada por el usuario
    indice_objetivo = df[(df['ano'] == ano) & (df['semana_epi'] == semana_epi)].index
    
    if len(indice_objetivo) == 0:
        raise ValueError(f"No existen registros en el dataset para el Año {ano} y Semana {semana_epi}")
        
    idx = indice_objetivo[0]
    
    # Validar la existencia de las 12 instancias anteriores requeridas para armar los rezagos
    if idx < 12:
        raise IndexError(f"La instancia en el índice {idx} no cuenta con 12 registros previos necesarios.")
        
    # Extraer la ventana de las 13 instancias en total (las 12 anteriores + la actual)
    ventana_historica = df.iloc[idx-12 : idx+1].reset_index(drop=True)
    
    # Obtener el mejor orden matemático desde el Excel
    orden_arimax = obtener_mejor_configuracion_arimax()
    
    # Separar variables exógenas sobrevivientes de Lasso
    columnas_no_exog = ['fecha', 'semana_epi', 'ano', 'casos_dengue']
    columnas_exog = [c for c in df.columns if c not in columnas_no_exog]
    
    # El histórico para ajustar el estado interno comprende las primeras 12 filas (t-12 a t-1)
    y_hist = ventana_historica['casos_dengue'].iloc[:-1]
    X_hist = ventana_historica[columnas_exog].iloc[:-1]
    
    # La fila número 13 (índice 12) representa el tiempo contemporáneo 't' a predecir
    X_futuro = ventana_historica[columnas_exog].iloc[[-1]]
    
    # Ajustar el ARIMAX lineal puntual
    model = sm.tsa.statespace.SARIMAX(
        y_hist, exog=X_hist, order=orden_arimax,
        enforce_stationarity=False, enforce_invertibility=False
    )
    model_fit = model.fit(disp=False)
    
    # Inferencia
    prediccion = model_fit.forecast(steps=1, exog=X_futuro).values[0]
    observado = ventana_historica['casos_dengue'].iloc[-1]
    
    casos_predichos = round(max(0.0, prediccion), 2)
    casos_observados = round(observado, 2)
    
    return casos_predichos, casos_observados


# =====================================================================
# 4. EJECUCIÓN AUTOMÁTICA DE PRUEBA
# =====================================================================
if __name__ == "__main__":
    # 1. Corre todo el entrenamiento, guarda el Excel plano y genera gráficos sin errores
    ejecutar_entrenamiento_arimax()
    
    # 2. Prueba de la función de usuario usando el nuevo dataset de la fase 08_sat
    print("\n>>> Ejecutando prueba de predicción puntual...")
    try:
        # Reemplazar por valores lógicos dentro del rango de tus datos reales
        ano_prueba = 2023
        semana_prueba = 30
        
        casos_p, casos_o = predecir_casos_dengue(ano_prueba, semana_prueba)
        print(f"\n[RESULTADO DEL MODELO]")
        print(f"Año: {ano_prueba} | Semana Epidemiológica: {semana_prueba}")
        print(f"-> Número de Casos Predichos: {casos_p}")
        print(f"-> Número de Casos Observados (Reales): {casos_o}")
    except Exception as e:
        print(f"Nota de la prueba: {e}")


>>> Iniciando entrenamiento de modelos ARIMAX...
>>> Entrenamiento completo. Métricas y gráficos guardados en: C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\3_lasso\3_resultados

>>> Ejecutando prueba de predicción puntual...

[RESULTADO DEL MODELO]
Año: 2023 | Semana Epidemiológica: 30
-> Número de Casos Predichos: 2.67
-> Número de Casos Observados (Reales): 1.0



# Ventajas de este Diseño:

* **Adiós al KeyError:** Al guardar el Excel con `ws.append(headers)` al inicio de la fila 1, eliminamos los títulos decorativos en la parte superior. De este modo, la función `pd.read_excel()` asocia los nombres de las columnas directamente al abrirse, eliminando cualquier desfase en las terminales.
* **Sin Expresiones Generadoras Conflictivas:** Se eliminó por completo el escaneo de filas dinámico. El script ahora indexa directamente el nombre de la columna objetivo, lo que anula el error de datos flotantes (`float is not iterable`).
* **Cumplimiento de Rutas SAT:** La función de usuario `predecir_casos_dengue` apunta y opera estrictamente bajo el archivo localizado en la ruta independiente del SAT (`08_sat\2_datos\1_raw\2_reducido_lasso_solo_dengue.csv`).